# Reto: Cadena de Tiendas Minoristas (RetailNow)
Análisis de ventas, inventarios y satisfacción del cliente con Pandas y NumPy.

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

In [2]:
# Carga de datos con rutas absolutas requeridas
path_ventas = '/workspace/ventas.csv'
path_inventarios = '/workspace/inventarios.csv'
path_satisfaccion = '/workspace/satisfaccion.csv'

df_ventas = pd.read_csv(path_ventas)
df_inventarios = pd.read_csv(path_inventarios)
df_satisfaccion = pd.read_csv(path_satisfaccion)

# Normalizar columnas
df_ventas.columns = df_ventas.columns.str.strip().str.lower()
df_inventarios.columns = df_inventarios.columns.str.strip().str.lower()
df_satisfaccion.columns = df_satisfaccion.columns.str.strip().str.lower()

# Limpieza: dropna()
df_ventas.dropna(inplace=True)
df_inventarios.dropna(inplace=True)
df_satisfaccion.dropna(inplace=True)

print('Formas tras limpieza:')
print(f'Ventas: {df_ventas.shape}, Inventarios: {df_inventarios.shape}, Satisfacción: {df_satisfaccion.shape}')

In [3]:
# Identificar columnas dinámicamente
col_tienda = [c for c in df_ventas.columns if 'tienda' in c or 'store' in c][0]
col_prod = [c for c in df_ventas.columns if 'prod' in c or 'item' in c][0]
col_cant = [c for c in df_ventas.columns if 'cant' in c or 'unidades' in c][0]
col_precio = [c for c in df_ventas.columns if 'prec' in c or 'price' in c][0]

# Ingresos totales por registro y por tienda
df_ventas['total_ventas'] = df_ventas[col_cant] * df_ventas[col_precio]
ventas_por_tienda = df_ventas.groupby(col_tienda)['total_ventas'].sum().reset_index()
ventas_por_tienda.rename(columns={'total_ventas': 'ingresos_totales'}, inplace=True)

# Resumen estadístico describe()
print('--- Resumen Estadístico de Ventas ---')
print(df_ventas[['total_ventas', col_cant, col_precio]].describe())
print('\n--- Ingresos Totales por Tienda ---')
print(ventas_por_tienda)

In [4]:
# Rotación de inventarios y filtro crítico (< 10%)
col_tienda_inv = [c for c in df_inventarios.columns if 'tienda' in c or 'store' in c][0]
col_prod_inv = [c for c in df_inventarios.columns if 'prod' in c or 'item' in c][0]
col_stock = [c for c in df_inventarios.columns if 'stock' in c or 'cant' in c][0]

ventas_unidades = df_ventas.groupby([col_tienda, col_prod])[col_cant].sum().reset_index()
df_inventarios = df_inventarios.merge(ventas_unidades, left_on=[col_tienda_inv, col_prod_inv], right_on=[col_tienda, col_prod], how='left')
df_inventarios[col_cant] = df_inventarios[col_cant].fillna(0)

resumen_inv = df_inventarios.groupby(col_tienda_inv).agg(total_vendido=(col_cant, 'sum'), total_stock=(col_stock, 'sum')).reset_index()
resumen_inv['tasa_rotacion'] = resumen_inv['total_vendido'] / resumen_inv['total_stock']

# Tiendas críticas: ventas < 10% del stock disponible
tiendas_criticas_inv = resumen_inv[resumen_inv['tasa_rotacion'] < 0.10]
print('--- Rotación de Inventarios ---')
print(resumen_inv)
print('\n--- Tiendas Críticas Inventario (< 10%) ---')
print(tiendas_criticas_inv)

In [5]:
# Satisfacción del cliente y filtro (< 60%)
col_tienda_sat = [c for c in df_satisfaccion.columns if 'tienda' in c or 'store' in c][0]
col_score_sat = [c for c in df_satisfaccion.columns if 'sat' in c or 'score' in c or 'punt' in c][0]

if df_satisfaccion[col_score_sat].max() <= 1.0:
    df_satisfaccion[col_score_sat] = df_satisfaccion[col_score_sat] * 100

promedio_sat = df_satisfaccion.groupby(col_tienda_sat)[col_score_sat].mean().reset_index()
df_rendimiento = ventas_por_tienda.merge(promedio_sat, left_on=col_tienda, right_on=col_tienda_sat, how='inner')

tiendas_baja_sat = df_rendimiento[df_rendimiento[col_score_sat] < 60.0]
print('--- Rendimiento vs Satisfacción ---')
print(df_rendimiento)
print('\n--- Tiendas con Satisfacción < 60% ---')
print(tiendas_baja_sat)

In [6]:
# Cálculos estadísticos con NumPy
ventas_array = df_ventas['total_ventas'].to_numpy()
mediana_ventas = np.median(ventas_array)
std_ventas = np.std(ventas_array, ddof=1)

print(f'Mediana de ventas: {mediana_ventas:.2f}')
print(f'Desviación estándar de ventas: {std_ventas:.2f}')

In [7]:
# Simulación de proyecciones con NumPy
np.random.seed(42)
num_tiendas = len(ventas_por_tienda)
dias_proyeccion = 30
promedio_diario = ventas_por_tienda['ingresos_totales'].mean() / 30
volatilidad = std_ventas / np.sqrt(len(ventas_array))

proyecciones = np.random.normal(loc=promedio_diario, scale=max(volatilidad, promedio_diario * 0.15), size=(num_tiendas, dias_proyeccion))
ventas_totales_proy = np.sum(proyecciones, axis=1)

media_proy = np.mean(ventas_totales_proy)
std_proy = np.std(ventas_totales_proy)

print(f'Media de ventas proyectadas: {media_proy:.2f}')
print(f'Desviación estándar proyectada: {std_proy:.2f}')